In [ ]:
# ---------------------------------------------
# Análisis distribuido de datos GPS con PySpark
# ---------------------------------------------

from pyspark.sql import SparkSession
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:

# Crear SparkSession
spark = SparkSession.builder.appName("AnalisisGPS").getOrCreate()

# Leer el CSV con datos simulados de GPS
df = spark.read.option("header", True).csv("viajes_gps.csv", inferSchema=True)
df.printSchema()

# Conteo de viajes por hora
df_horas = df.groupBy("hora_salida").count().orderBy("hora_salida")
df_horas_pd = df_horas.toPandas()

# Conteo de rutas más utilizadas
df_rutas = df.withColumn("ruta", df["zona_origen"] + " → " + df["zona_destino"])
df_rutas = df_rutas.groupBy("ruta").count().withColumnRenamed("count", "num_viajes")
df_rutas_pd = df_rutas.toPandas()

# Conteo de entradas y salidas por zona
entradas = df.groupBy("zona_destino").count().withColumnRenamed("count", "total_entradas")
salidas = df.groupBy("zona_origen").count().withColumnRenamed("count", "total_salidas")

df_zonas = entradas.join(salidas, entradas.zona_destino == salidas.zona_origen, "outer") \
                   .withColumn("zona", entradas["zona_destino"]) \
                   .select("zona", "total_entradas", "total_salidas") \
                   .fillna(0)

df_zonas_pd = df_zonas.toPandas()

# Exportar resultados
df_horas_pd.to_csv("viajes_por_hora.csv", index=False)
df_rutas_pd.to_csv("top_rutas.csv", index=False)
df_zonas_pd.to_csv("zonas_entrada_salida.csv", index=False)

In [ ]:

# -------------------------
# Visualizaciones con Pandas
# -------------------------

sns.set(style="whitegrid")


In [ ]:

# 1. Viajes por hora
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_horas_pd, x="hora_salida", y="count", marker="o")
plt.title("Cantidad de viajes por hora del día")
plt.xlabel("Hora del día")
plt.ylabel("Número de viajes")
plt.grid(True)
plt.tight_layout()
plt.savefig("grafico_viajes_por_hora.png")
plt.close()

In [ ]:

# 2. Rutas con mayor congestión
plt.figure(figsize=(10, 6))
df_rutas_sorted = df_rutas_pd.sort_values("num_viajes", ascending=False)
sns.barplot(data=df_rutas_sorted.head(10), x="num_viajes", y="ruta", palette="magma")
plt.title("Top rutas con mayor número de viajes")
plt.xlabel("Número de viajes")
plt.ylabel("Ruta")
plt.tight_layout()
plt.savefig("grafico_rutas_congestionadas.png")
plt.close()


In [ ]:

# 3. Entradas y salidas por zona
df_zonas_melted = df_zonas_pd.melt(id_vars="zona", value_vars=["total_entradas", "total_salidas"],
                                   var_name="tipo", value_name="cantidad")


In [ ]:


plt.figure(figsize=(10, 6))
sns.barplot(data=df_zonas_melted, x="zona", y="cantidad", hue="tipo", palette="Set2")
plt.title("Entradas y salidas por zona")
plt.xlabel("Zona")
plt.ylabel("Cantidad de vehículos")
plt.tight_layout()
plt.savefig("grafico_entrada_salida_zonas.png")
plt.close()

# Finalizar sesión de Spark
spark.stop()
